# 04 · Regularization and how to choose it

Chapter 03 ended at an impasse: many wildly different slip models fit the data
equally well, and the best-fitting one was nonsense. The missing ingredient is
information from outside the data, such as the expectation that slip varies
smoothly across a fault. **Regularization** is the standard way to state that
information and blend it with the data.

This chapter has two halves. The first half shows what regularization is and
what assumptions each flavor encodes. The second half faces the question every
practitioner must answer: how strongly should the extra information be
weighted? We will meet three principled ways to decide.

**Learning objectives**

By the end of the chapter, you will be able to:

- write the regularized least-squares objective and explain each term
- state the assumptions behind smoothing, damping, and stress-based penalties
- explain regularization as extra equations and as a prior belief
- run smoothed inversions with `geodef.solve` and interpret a strength sweep
- choose the regularization strength with the L-curve, ABIC, and
  cross-validation, and compare their answers

**Prerequisites:** Chapter 03
**Estimated time:** 90–120 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Adding a second opinion to the misfit

Least squares asked only one question: how well does the model fit the data?
Regularization adds a second question: how reasonable is the model on its own
terms? Both are scored, and the inversion minimizes their sum:

$$ \min_{\mathbf{m}} \;
\underbrace{\lVert G\mathbf{m} - \mathbf{d}\rVert^2_{C_d}}_{\text{fit the data}}
\;+\;
\underbrace{\lambda\,\lVert L(\mathbf{m} - \mathbf{m}_{\text{ref}})\rVert^2}_{\text{stay reasonable}}. $$

Three new symbols appear, and each one is a scientific choice:

- $L$ is the **regularization operator**. It measures what "unreasonable"
  means: large slip, rough slip, or something else.
- $\lambda$ is the **regularization strength**. It sets the exchange rate
  between the two scores. As $\lambda \to 0$ we recover the unregularized
  catastrophe of Chapter 03; as $\lambda \to \infty$ the data are ignored and
  $\mathbf{m}$ is pushed all the way to $\mathbf{m}_{\text{ref}}$.
- $\mathbf{m}_{\text{ref}}$ is a **reference model**, the slip we would prefer
  in the absence of data. The default is zero.

This construction is called **Tikhonov regularization**.

## 2. Choosing what to penalize

GeoDef provides three ready-made operators, each encoding a different belief:

- **Smoothing** (`regularization="laplacian"`). $L$ is the discrete Laplacian,
  which measures how much each patch differs from the average of its
  neighbors; in one dimension it is the pattern $m_{i-1} - 2m_i + m_{i+1}$.
  Penalizing it favors slip that varies gradually across the fault. A large,
  smooth asperity costs almost nothing; a patch-to-patch checkerboard costs a
  fortune.
- **Damping** (`regularization="damping"`). $L$ is the identity matrix, so the
  penalty is simply the size of the slip itself. This favors the smallest
  model that fits the data and tends to shrink the total moment.
- **Stress kernel** (`regularization="stresskernel"`). $L$ is built from the
  elastic stress interactions between patches, a physically motivated cousin
  of smoothing.

None of these is "correct." Each is an assumption, and an honest study states
which one it used and why.

## 3. Regularization is just extra equations

There is a satisfying mechanical view of all this. Stack the penalty
underneath the data equations as if it were more data:

$$ \begin{bmatrix} G \\ \lambda L \end{bmatrix} \mathbf{m}
   = \begin{bmatrix} \mathbf{d} \\ \lambda L\,\mathbf{m}_{\text{ref}} \end{bmatrix}. $$

Solving this taller system by ordinary least squares gives exactly the
regularized solution. The prior enters as synthetic observations ("the
Laplacian of the slip is zero, please") with weight $\lambda$, competing on
equal terms with the real observations. Nothing mysterious happens inside the
solver; we simply told it more things.

## 4. Two more ways to understand the same formula

**As a noise filter.** Chapter 03 explained the failure of least squares
through slip patterns the data barely see. Damping-style regularization
multiplies each of those patterns by a factor between 0 and 1 that depends on
how visible the pattern is: clearly observed patterns pass through almost
untouched, while poorly observed ones (the noise amplifiers) are turned down
first. Regularization does not restore information the data lack; it stops the
inversion from pretending that information exists.

**As a prior belief.** The same formula is what you get by treating the
penalty as a Gaussian probability distribution over models before seeing any
data, and then asking for the most probable model afterward. In that language,
$L$ describes which models are believed plausible and $\lambda$ describes how
firmly. Chapter 14 builds a full Bayesian treatment on this idea.

Both views carry the same warning: regularization trades noise sensitivity for
**bias**. A smoothed solution is stabler than the raw one, and also
systematically smoother than the truth.

## 5. Rebuild the shared scenario

Chapters 03 through 10 reuse one synthetic scenario so that every method faces
the same target: a 12 by 6 patch megathrust, a smooth 3 m thrust asperity, an
8 by 8 GNSS grid, and 1 cm of seeded noise. The next cells rebuild it exactly
as in Chapter 03; see that chapter for the step-by-step narration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The chapter 03 megathrust: 12 x 6 patches, centered 25 km deep.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape

In [ ]:
# True slip: a smooth thrust asperity near the fault center, dip slip only.
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

In [ ]:
# An 8 x 8 GNSS grid; synthetic data with 1 cm of seeded Gaussian noise.
grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
print(f"{N} patches, {n_stations} stations, {d_obs.size} observations")

## 6. One smoothed inversion

Adding regularization to `geodef.solve` takes two arguments: the operator name
and the strength. Let us solve once without regularization and once with
Laplacian smoothing at $\lambda = 1$, and compare both to the truth.

In [ ]:
raw = geodef.solve(fault, gnss, components="dip")
smooth = geodef.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=1.0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = slip_true[N:].max()
limit_raw = abs(raw.dip_slip).max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-limit_raw, vmax=limit_raw, title="Unregularized",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, smooth.dip_slip, ax=axes[2], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Laplacian, lambda = 1",
                 colorbar_label="Slip (m)")
plt.tight_layout()

The smoothed inversion recovers the asperity. It does not reproduce the truth
exactly; some detail is genuinely unresolvable from noisy surface data, and no
honest method will conjure it back. But the result is smooth, positive where
it should be, and physically plausible, everything the middle panel is not.

## 7. The effect of the strength

How much smoothing is the right amount? Sweep $\lambda$ across four orders of
magnitude and watch the solution change character. Each solve is one line; we
collect the results in a list.

In [ ]:
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0]
recovered = []
chi_squared = []
for lam in lambdas:
    result = geodef.solve(fault, gnss, components="dip",
                          regularization="laplacian",
                          regularization_strength=lam)
    recovered.append(result.dip_slip)
    chi_squared.append(result.reduced_chi2)

First the recovered slip, on one shared color scale so amplitudes are directly
comparable.

In [ ]:
truth = slip_true[N:]
vmax = truth.max()
fig, axes = plt.subplots(1, 6, figsize=(16.5, 3), constrained_layout=True)
geodef.plot.slip(fault, truth, ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
for ax, lam, slip, chi2 in zip(axes[1:], lambdas, recovered, chi_squared):
    geodef.plot.slip(fault, slip, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"lambda = {lam:g}\nchi2 = {chi2:.2f}")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8, label="Slip (m)")

Now subtract the truth from each panel. The error maps show *how* each extreme
fails, not just that it fails.

In [ ]:
errors = [slip - truth for slip in recovered]
emax = max(np.abs(error).max() for error in errors)
fig, axes = plt.subplots(1, 5, figsize=(14, 3), constrained_layout=True)
for ax, lam, error in zip(axes, lambdas, errors):
    geodef.plot.slip(fault, error, ax=ax, cmap="RdBu_r",
                     vmin=-emax, vmax=emax, colorbar=False,
                     title=f"error, lambda = {lam:g}")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8,
             label="Recovered - true (m)")

The two failure modes look different. At small $\lambda$ the error is noisy
and alternates in sign: leftover overfitting. At large $\lambda$ the error is
broad and one-sided: the asperity has been flattened, its peak lost and its
edges smeared outward, and the reduced chi-squared climbs above 1 because the
over-smoothed model can no longer fit the signal. The best compromise lies
between, and the rest of this chapter is about finding it without peeking at
the true answer.

## 8. Damping versus smoothing

Before choosing the strength, it is worth seeing what the choice of *operator*
does. Damping pulls slip toward zero everywhere, so it prefers a small model;
smoothing penalizes roughness but is perfectly happy with a large, smooth
bump. At a gentle strength the two barely differ here, because our true model
happens to be both smooth *and* compact. To make the disagreement visible we
turn both priors up hard, to $\lambda = 30$.

In [ ]:
smooth = geodef.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=30.0)
damped = geodef.solve(fault, gnss, components="dip",
                      regularization="damping", regularization_strength=30.0)
print(f"smoothing: peak slip {smooth.dip_slip.max():.2f} m, Mw {smooth.Mw:.2f}")
print(f"damping:   peak slip {damped.dip_slip.max():.2f} m, Mw {damped.Mw:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, smooth.dip_slip, ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Smoothing (Laplacian), lambda = 30",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, damped.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Damping (minimum norm), lambda = 30",
                 colorbar_label="Slip (m)")
plt.tight_layout()

Now the two priors part ways. Damping has shrunk the peak more aggressively
than smoothing, because shrinking slip everywhere is exactly its goal; the
smoothed model keeps a taller, rounder bump. Neither result is wrong, and at
this over-tuned strength neither is a good fit. The point is only that the two
priors ask differently phrased questions, which is why reporting "we used
regularization" without naming the operator leaves out half the story.

## 9. Choosing the strength: the L-curve

Every $\lambda$ produces one solution, which we can score twice: how badly it
misfits the data, and how rough it is,

$$ \rho(\lambda) = \lVert G\hat{\mathbf m}_\lambda - \mathbf d\rVert,
   \qquad
   \eta(\lambda) = \lVert L\hat{\mathbf m}_\lambda\rVert . $$

Plotting roughness against misfit on logarithmic axes as $\lambda$ sweeps
traces out a characteristic L-shaped curve. Its steep branch is the
overfitting regime: roughness explodes while the misfit barely improves. Its
flat branch is the over-smoothing regime: the misfit grows with nothing gained
in return. The **corner** joining the branches is the natural compromise,
where the curve bends most sharply, and `geodef.invert.lcurve` sweeps
$\lambda$, locates that point of maximum curvature, and marks it.

In [ ]:
lc = geodef.invert.lcurve(fault, gnss, components="dip",
                          regularization="laplacian",
                          regularization_range=(1e-2, 1e3), n=40)
lc.plot()
print(f"L-curve corner: lambda = {lc.optimal:.3g}")

## 10. Choosing the strength: ABIC

The **Akaike Bayesian Information Criterion** comes from the prior-belief view
of Section 4. Treat the smoothing penalty as a Gaussian prior on slip, with
$\lambda$ setting how tight that prior is. For any candidate $\lambda$ one can
ask: averaged over every slip model the prior allows, how probable are the
data we actually observed? ABIC is (minus twice the logarithm of) that
probability, so the **minimum** of the ABIC curve marks the most supported
strength.

The averaging is what gives ABIC its balance. A prior that is too loose wastes
probability on wild models the data reject; one that is too tight excludes the
models the data favor. Both extremes score poorly, an automatic version of
Occam's razor. The formula (Fukuda and Johnson, 2008, in the further reading)
works out to a closed-form expression for linear problems, and
`geodef.invert.abic_curve` evaluates it across a sweep. Passing
`regularization_strength="abic"` to `geodef.solve` selects the minimizer
directly.

In [ ]:
ab = geodef.invert.abic_curve(fault, gnss, components="dip",
                              regularization="laplacian",
                              regularization_range=(1e-2, 1e3), n=40)
ab.plot()
print(f"ABIC minimum: lambda = {ab.optimal:.3g}")

## 11. Choosing the strength: cross-validation

**Cross-validation** asks the most pragmatic question of the three: which
strength best predicts data the inversion never saw? Split the observations
into $k$ groups, or folds. For each fold in turn, fit the model using only the
other $k-1$ folds, predict the held-out fold, and record the prediction error.
The $\lambda$ with the smallest total held-out error wins.

Cross-validation needs no prior story and no geometric heuristic, only the
data's own predictive performance. The price is computation: every trial
$\lambda$ costs $k$ inversions. Passing `regularization_strength="cv"` (with
`cv_folds=k`) makes `geodef.solve` run the whole procedure internally.

In [ ]:
cv = geodef.solve(fault, gnss, components="dip",
                  regularization="laplacian",
                  regularization_strength="cv", cv_folds=5)
print(f"cross-validation choice: lambda = {cv.regularization_strength:.3g}")

## 12. Do the three criteria agree?

The three criteria answer differently phrased questions, so they need not land
on the same number, but on a well-behaved problem they usually agree to within
a factor of a few. Let us solve at each recommended strength and compare.

In [ ]:
choices = {
    "L-curve": lc.optimal,
    "ABIC": ab.optimal,
    "cross-val": cv.regularization_strength,
}
solutions = {}
for name, lam in choices.items():
    result = geodef.solve(fault, gnss, components="dip",
                          regularization="laplacian",
                          regularization_strength=lam)
    solutions[name] = result.dip_slip

In [ ]:
truth = slip_true[N:]
vmax = truth.max()
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), constrained_layout=True)
geodef.plot.slip(fault, truth, ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
for ax, (name, slip) in zip(axes[1:], solutions.items()):
    geodef.plot.slip(fault, slip, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"{name}\nlambda = {choices[name]:.2g}")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8, label="Slip (m)")

The slip maps are hard to tell apart by eye. Subtracting the truth confirms
it: the three error maps are nearly identical, so on this problem the choice
of criterion matters far less than the choice to regularize at all.

In [ ]:
differences = {name: slip - truth for name, slip in solutions.items()}
dmax = max(np.abs(diff).max() for diff in differences.values())
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), constrained_layout=True)
for ax, (name, diff) in zip(axes, differences.items()):
    geodef.plot.slip(fault, diff, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"{name} - true")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8,
             label="Recovered - true (m)")

**Checkpoint.** Cross-validation held stations out at random. Why might that
be too optimistic for InSAR data, where Chapter 06 shows that neighboring
pixels share correlated noise?

<details><summary>Answer</summary>

If a held-out pixel's neighbors are in the training set, and all of them share
the same smooth atmospheric error, then predicting the held-out pixel is
partly a matter of reproducing that shared error rather than the fault signal.
The held-out score looks better than genuine prediction would be, biasing the
choice toward under-smoothing. Folds that hold out spatial blocks reduce the
leakage.

</details>

## Checkpoint questions

**What do $L$ and $\lambda$ control, separately?**

<details><summary>Answer</summary>

$L$ defines *what* is penalized: which property of the model (size, roughness,
stress) counts as unreasonable. $\lambda$ defines *how strongly* that penalty
competes with the data misfit. Changing $L$ changes the kind of solution;
changing $\lambda$ moves along a family of solutions of the same kind.

</details>

**Why can the L-curve, ABIC, and cross-validation give different answers
without any of them being wrong?**

<details><summary>Answer</summary>

They optimize different criteria: geometric balance between misfit and
roughness, probability of the data under a Gaussian prior, and out-of-sample
prediction error. Each is the right answer to its own question; the questions
simply are not identical.

</details>

**Does a larger $\lambda$ always mean a smoother-looking model?**

<details><summary>Answer</summary>

Only within one operator. Across operators the comparison is meaningless:
damping at large $\lambda$ gives a *small* model, not a smooth one, and the
numerical scale of $\lambda$ depends on how $L$ is normalized.

</details>

## Common mistakes

- **Reporting the operator without the strength, or vice versa.** A regularized
  result is reproducible only if both choices (and the reference model, if
  any) are stated. Beware also of conventions that write the strength as
  $\lambda^2$: passing a squared value to an API expecting $\lambda$ changes
  the intended weight.
- **Calling damping "smoothing."** Damping encodes a minimum-size assumption
  and suppresses moment. Labeling it smoothing hides an assumption that
  directly affects the headline magnitude.
- **Choosing $\lambda$ by eye from the slip maps alone.** Picking the map that
  "looks geological" and then quoting the criterion that happens to agree is
  circular. Decide on the selection criterion first.
- **Expecting regularization to add resolution.** It suppresses unresolved
  patterns; it cannot recover them. Fine structure absent from the smoothed
  model may be smoothing bias, not evidence of absence. Chapter 08 gives the
  tools to tell.

## Recap

- Regularization adds a stated preference, encoded by an operator $L$, a
  strength $\lambda$, and a reference model, to the data misfit.
- It is equivalently extra equations in an augmented system, a filter on
  poorly observed slip patterns, and a Gaussian prior belief.
- Smoothing and damping encode different assumptions and give visibly
  different models at the same misfit.
- The L-curve, ABIC, and cross-validation choose $\lambda$ by geometry,
  marginal probability, and prediction; on this problem all three land close
  together.

Chapter 05 brings in a second, very different dataset and shows how to weight
the two against each other.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/04_regularization_solutions.ipynb`.

1. **Same misfit, different beliefs.** Tune the damping strength until its
   reduced chi-squared matches the Laplacian solution at $\lambda = 1$, then
   compare the two slip models. What differs when the fit does not?
2. **A nonzero reference model.** Solve with a uniform 1 m dip-slip reference
   model at several strengths. Which patches follow the reference, and which
   stay pinned by the data?
3. **Noisier data, new strengths.** Double the noise level (in both the data
   and the stated uncertainties) and rerun the L-curve, ABIC, and
   cross-validation. Do the recommended strengths move in the direction you
   expected?
4. **Challenge: build your own operator.** Construct a first-difference
   operator along strike (penalizing $m_{i+1} - m_i$), pass it to the
   inversion, and select its strength with ABIC. Compare the result with the
   Laplacian solution.

## Further reading

- Tikhonov, A. N., and Arsenin, V. Y. (1977), *Solutions of Ill-Posed
  Problems*, the original regularization theory.
- Hansen, P. C. (1992), "Analysis of discrete ill-posed problems by means of
  the L-curve," *SIAM Review*, 34(4), 561–580.
- Fukuda, J., and Johnson, K. M. (2008), "A fully Bayesian inversion for
  spatial distribution of fault slip with objective smoothing," *Bulletin of
  the Seismological Society of America*, 98(3), 1128–1146, for ABIC in
  geodetic inversion.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, Chapter 12 on slip
  inversion practice.
- [GeoDef inversion documentation](../docs/invert.md) for the full
  regularization and hyperparameter interfaces.
- The next course chapter is `05_multiple_datasets.ipynb`.